# <h1 align="center"><font color="red">Practical Guide to Using ``ChromaDB`` for ``RAG`` and Semantic Search</font></h1>

<font color="pink">Senior Data Scientist.: Dr. Eddy Giusepe Chirinos Isidro</font>

Este notebook, baseado no tutorial de [Alpha Iterations](https://ai.plainenglish.io/build-agentic-rag-using-langgraph-b568aa26d710), demonstra como criar e consultar coleções ``ChromaDB`` com ambos ``SentenceTransformers`` e ``OpenAI embeddings``. Inclui exemplos para: adicionar documentos, usar funções de embedding, realizar consultas de similaridade e usar ``OpenAI`` para geração.


## <font color="gree">Configuração e Inicialização do Client ``Chroma``</font>

A seguir vamos a instalar as dependências mínimas usadas neste notebook. Execute as seguintes comandos para preparar o ambiente.

```bash
pip install sentence-transformers openai chromadb

ou 

uv add sentence-transformers openai chromadb
```

A próxima célula mostra como criar um cliente ``Chroma``, tanto em memória quanto persistente. Use ``PersistentClient`` para manter uma cópia local do banco de dados no disco.

In [ ]:
import chromadb

# Opção 1: Usando configurações padrão (em memória)
client = chromadb.Client()

# Com distância de cosseno (recomendado para embeddings de texto)
collection_cosine = client.get_or_create_collection(
    name="my_docs_cosine",
    metadata={"hnsw:space": "cosine"}
)

# Com distância Euclidiana L2 (padrão)
collection_l2 = client.get_or_create_collection(
    name="my_docs_l2",
    metadata={"hnsw:space": "l2"}
)

# Com produto interno
collection_ip = client.get_or_create_collection(
    name="my_docs_ip",
    metadata={"hnsw:space": "ip"}
)


In [ ]:
# Opção 2: Usando armazenamento local persistente (Recomendado para aplicações reais)
client = chromadb.PersistentClient(path="./chroma_db")

# Criar ou obter uma coleção com similaridade de cosseno
collection = client.get_or_create_collection(
    name="my_docs",
    metadata={"hnsw:space": "cosine"}  # Usando distância de cosseno (varia de 0 a 2)
)

## <font color="gree">Adicionar Documentos a uma Coleção</font>

A célula abaixo mostra um pequeno conjunto de dados de exemplo e como adicionar ``documentos``, ``metadados`` e ``ids`` a uma coleção ``Chroma``. Use chaves de metadados significativas para habilitar buscas filtradas (o parâmetro ``where`` nas consultas).

``Dica:`` Para conjuntos de dados maiores, carregue documentos de arquivos ou um banco de dados, em vez de incorporá-los diretamente no notebook.

In [ ]:
documents = [
    "As vacinas contra COVID-19 demonstraram reduzir casos graves e taxas de hospitalização.",
    "A gerenciamento da diabetes requer monitoramento regular dos níveis de glicose sanguínea e ajustes no estilo de vida.",
    "A hipertensão é um fator de risco principal para doenças cardíacas e acidentes vasculares, requerendo intervenção oportuna.",
    "Os exames de ressonância magnética fornecem imagens detalhadas de tecidos moles, auxiliando no diagnóstico de doenças neurológicas.",
    "A resistência à antibióticos é um problema crescente no tratamento de infecções bacterianas.",
    "A telemedicina permite que pacientes consultem médicos remotamente, melhorando o acesso aos serviços de saúde.",
    "A imunoterapia do câncer utiliza o sistema imunológico do corpo para alcançar e destruir células malignas.",
    "As doenças mentais, como depressão e ansiedade, afetam milhões e requerem cuidados abrangentes.",
    "Os dispositivos Wearable podem rastrear taxas de batimento cardíaco, padrões de sono e níveis de atividade para insights de saúde personalizados.",
    "A genética pode ajudar a identificar indivíduos em risco para doenças hereditárias e orientar estratégias preventivas."
]

# Metadata para filtragem/busca:
metadatas = [
    {"topic": "vaccines", "source": "WHO"},
    {"topic": "diabetes", "source": "CDC"},
    {"topic": "cardiology", "source": "MayoClinic"},
    {"topic": "diagnostics", "source": "JohnsHopkins"},
    {"topic": "infectious_disease", "source": "CDC"},
    {"topic": "telemedicine", "source": "NIH"},
    {"topic": "oncology", "source": "NCI"},
    {"topic": "mental_health", "source": "WHO"},
    {"topic": "wearables", "source": "Stanford"},
    {"topic": "genetics", "source": "GenomicsInstitute"}
]

# IDs únicos para cada documento:
ids = [f"doc{i + 1}" for i in range(len(documents))]

# Exemplo: Adicionando a uma coleção Chroma
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)


### <font color="orange">Inspecionando Dados Armazenados no ChromaDB</font>

**Importante:** O arquivo `chroma.sqlite3` é um banco de dados **binário** e não pode ser aberto diretamente como texto. Para visualizar os dados, use o ChromaDB através do Python:


In [ ]:
# 📊 Inspecionando o conteúdo da collection

# Obter informações básicas
print(f"📁 Nome da collection: {collection.name}")
print(f"📈 Total de documentos: {collection.count()}")
print(f"🔧 Metadados da collection: {collection.metadata}")
print()

# Listar todos os documentos (peek = espiar)
dados = collection.peek(limit=5)  # Mostra os primeiros 5 documentos
print("🔍 Primeiros 5 documentos:")
print(f"IDs: {dados['ids']}")
print(f"Documentos: {dados['documents']}")
print(f"Metadados: {dados['metadatas']}")
print()

# Obter TODOS os documentos (use com cuidado em collections grandes!)
todos_dados = collection.get()
print(f"📚 Total de documentos recuperados: {len(todos_dados['ids'])}")
print(f"🏷️ IDs: {todos_dados['ids']}")
print(f"📄 Primeiro documento: {todos_dados['documents'][0]}")


In [ ]:
query = "Como funciona a imunoterapia do câncer?"

results = collection.query(
    query_texts=[query],
    n_results=2 #top k results
)

results

In [ ]:
query = "Como podem os dispositivos Wearable rastrear a saúde do coração?"

results = collection.query(
    query_texts=[query],
    n_results=3,
    where={"topic": "cardiology"}  # Filtro por metadados
)

results

## <font color="gree">Embeddings com ``SentenceTransformers``</font>

Esta seção demonstra o uso de uma função de incorporação de ``SentenceTransformer`` com ``Chroma``. Usamos ``all-MiniLM-L6-v2`` para um equilíbrio de ``velocidade`` e ``precisão``. Se você precisar de uma qualidade melhor, troque por um modelo maior.

In [ ]:
# Import Chroma and embedding function
import chromadb
from chromadb.utils import embedding_functions

# ✅ Define o modelo de Embedding (Sentence Transformer)
# all-MiniLM-L6-v2 é leve, rápido e bom para a busca de texto semântico médico
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# ✅ Inicializa um cliente Chroma persistente (dados armazenados no disco)
client = chromadb.PersistentClient(path="./medical_chroma_db")

# ✅ Cria (ou obtém) uma coleção com suporte a Embedding
collection = client.get_or_create_collection(
    name="medical_documents",
    embedding_function=sentence_transformer_ef
)

# ✅ Medical domain documents
documents = [
    "As vacinas contra COVID-19 demonstraram reduzir casos graves e taxas de hospitalização.",
    "A gerenciamento da diabetes requer monitoramento regular dos níveis de glicose sanguínea e ajustes no estilo de vida.",
    "A hipertensão é um fator de risco principal para doenças cardíacas e acidentes vasculares, requerendo intervenção oportuna.",
    "Os exames de ressonância magnética fornecem imagens detalhadas de tecidos moles, auxiliando no diagnóstico de doenças neurológicas.",
    "A resistência à antibióticos é um problema crescente no tratamento de infecções bacterianas.",
    "A telemedicina permite que pacientes consultem médicos remotamente, melhorando o acesso aos serviços de saúde.",
    "A imunoterapia do câncer utiliza o sistema imunológico do corpo para alcançar e destruir células malignas.",
    "As doenças mentais, como depressão e ansiedade, afetam milhões e requerem cuidados abrangentes.",
    "Os dispositivos Wearable podem rastrear taxas de batimento cardíaco, padrões de sono e níveis de atividade para insights de saúde personalizados.",
    "A genética pode ajudar a identificar indivíduos em risco para doenças hereditárias e orientar estratégias preventivas."
]

# ✅ Correspondente metadata para filtragem/busca
metadatas = [
    {"topic": "vaccines", "source": "WHO"},
    {"topic": "diabetes", "source": "CDC"},
    {"topic": "cardiology", "source": "MayoClinic"},
    {"topic": "diagnostics", "source": "JohnsHopkins"},
    {"topic": "infectious_disease", "source": "CDC"},
    {"topic": "telemedicine", "source": "NIH"},
    {"topic": "oncology", "source": "NCI"},
    {"topic": "mental_health", "source": "WHO"},
    {"topic": "wearables", "source": "Stanford"},
    {"topic": "genetics", "source": "GenomicsInstitute"}
]

# ✅ IDs únicos para cada documento
ids = [f"doc{i + 1}" for i in range(len(documents))]

# ✅ Adicionar dados à coleção Chroma
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print("✅ Documentos médicos adicionados com sucesso à Chroma!")


In [ ]:
query = "Como podem os dispositivos Wearable ajudar com a saúde do coração?"

results = collection.query(
    query_texts=[query],
    n_results=2
)

print("🔍 Resultados da query:")
results

In [ ]:
from sentence_transformers import SentenceTransformer

# Use o mesmo modelo usado para Embedding de coleção
model = SentenceTransformer("all-MiniLM-L6-v2")

query = "Como podem os dispositivos Wearable ajudar com a saúde do coração?"
# Encode retorna um array numpy para uma lista de entradas; converta para lista de listas
query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,  # Passa Embeddings diretamente
    n_results=2
)

print("🔍 Resultados da query (embedding manual):")
results

## <font color="gree">Usando Embeddings OpenAI (manual)</font>

Os seguintes exemplos usam a ``API de Embedding do OpenAI``. Antes de executá-los, defina sua chave API no ambiente (por exemplo, em um arquivo ``.env``) e nunca comita segredos a controle de versão.

In [ ]:
# Salvar documentos médicos na Chroma usando Embeddings OpenAI manualmente
import chromadb
import os
from dotenv import load_dotenv, find_dotenv
#import openai
from openai import OpenAI

# ✅ Carregar variáveis de ambiente:
_ = load_dotenv(find_dotenv())  # read local .env file
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

# ✅ Configurar chave API OpenAI:
clientOpenAI = OpenAI(api_key=OPENAI_API_KEY)
#openai.api_key = OPENAI_API_KEY

# ✅ Função personalizada para gerar embeddings usando API OpenAI:
def generate_embedding(text: str, model: str = "text-embedding-3-small") -> list:
    """
    Gera um vetor de Embedding para um texto fornecido usando a API de Embeddings da OpenAI.
    """
    response = clientOpenAI.embeddings.create(
        input=text,
        model=model
    )
    embedding_vector = response.data[0].embedding
    return embedding_vector

# ✅ Inicializar um cliente Chroma persistente:
client = chromadb.PersistentClient(path="./medical_chroma_openai_db")

# ✅ Criar ou obter coleção (sem função de embedding Chroma):
collection = client.get_or_create_collection(
    name="medical_documents_custom"
)

# ✅ Documentos do domínio médico:
documents = [
    "As vacinas contra COVID-19 demonstraram reduzir casos graves e taxas de hospitalização.",
    "A gerenciamento da diabetes requer monitoramento regular dos níveis de glicose sanguínea e ajustes no estilo de vida.",
    "A hipertensão é um fator de risco principal para doenças cardíacas e acidentes vasculares, requerendo intervenção oportuna.",
    "Os exames de ressonância magnética fornecem imagens detalhadas de tecidos moles, auxiliando no diagnóstico de doenças neurológicas.",
    "A resistência à antibióticos é um problema crescente no tratamento de infecções bacterianas.",
    "A telemedicina permite que pacientes consultem médicos remotamente, melhorando o acesso aos serviços de saúde.",
    "A imunoterapia do câncer utiliza o sistema imunológico do corpo para alcançar e destruir células malignas.",
    "As doenças mentais, como depressão e ansiedade, afetam milhões e requerem cuidados abrangentes.",
    "Os dispositivos Wearable podem rastrear taxas de batimento cardíaco, padrões de sono e níveis de atividade para insights de saúde personalizados.",
    "A genética pode ajudar a identificar indivíduos em risco para doenças hereditárias e orientar estratégias preventivas."
]

# ✅ Corresponding metadata
metadatas = [
    {"topic": "vaccines", "source": "WHO"},
    {"topic": "diabetes", "source": "CDC"},
    {"topic": "cardiology", "source": "MayoClinic"},
    {"topic": "diagnostics", "source": "JohnsHopkins"},
    {"topic": "infectious_disease", "source": "CDC"},
    {"topic": "telemedicine", "source": "NIH"},
    {"topic": "oncology", "source": "NCI"},
    {"topic": "mental_health", "source": "WHO"},
    {"topic": "wearables", "source": "Stanford"},
    {"topic": "genetics", "source": "GenomicsInstitute"}
]

# ✅ IDs únicos:
ids = [f"doc{i + 1}" for i in range(len(documents))]

# ✅ Gerar embeddings manualmente:
embeddings = [generate_embedding(doc) for doc in documents]

# ✅ Adicionar dados à coleção Chroma com embeddings pré-computados:
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
    embeddings=embeddings
)

print("✅ Documentos médicos adicionados com sucesso à Chroma (embeddings personalizados)!")

# ✅ Exemplo de query embedding:
query = "Como podem os dispositivos Wearable ajudar com a saúde do coração?"
query_embedding = generate_embedding(query)

# ✅ Realizar busca de similaridade usando o embedding da query
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2
)

print("🔍 Query Results:")
results

## <font color="gree">Usando ChromaDB com LLM como parte do sistema RAG</font>

In [ ]:
from openai import OpenAI

_ = load_dotenv(find_dotenv())  # read local .env file
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

client_llm = OpenAI(api_key=OPENAI_API_KEY)

query = "Como podem os dispositivos Wearable ajudar com a saúde do coração?"

query_embedding = generate_embedding(query)

results = collection.query(query_embeddings=[query_embedding], n_results=5)
context = "\n".join(results["documents"][0])

prompt = f"""
Responda de forma clara, objetiva e sucinta a seguinte pergunta usando o contexto abaixo.

Contexto:
{context}

Pergunta: {query}
"""

response = client_llm.chat.completions.create(
    model="gpt-5-nano", #"gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)


print(response.choices[0].message.content)